In [1]:

import time

from generate_recipe_task import *
from testing_old.experiment_classes import WrongIngTestOld

from together import Together

recipe_df = pd.read_csv("../csv/over40ing.csv", header=None)

num_steps = [len(row.iloc[5].split('\n')) for i, row in recipe_df.iterrows()]
num_ings = [len(row.iloc[3].split('\n')) for i, row in recipe_df.iterrows()]

recipe_df['num_steps'] = num_steps
recipe_df['num_ings'] = num_ings

long_recipes = recipe_df[recipe_df['num_steps'] >= 5]

long_recipes

KeyError: 'instructions'

In [2]:
num_recipes = 5

actor_specs = ["Alice", "She", 1, 0.5, 0]
transcipt_steps = 3
max_recipes = 4

In [9]:
selected_row = long_recipes.iloc[3]
print(f'Generating transcript for {selected_row.iloc[0]}')
generate_task_file_from_row(selected_row, actor_specs, transcipt_steps)

Generating transcript for Land And Sea White Meat Version Of Surf And Turf
Traceback (most recent call last):
  File "/Users/jerryliu/Desktop/workspace/cookingLlama/recipe_task/recipe_task.py", line 54, in <module>
    t1 = mix('mix', 'Mix all ingredients', [apricot, nectarine, redpepper, jalapeno, garlic, raisins, onionsA, cilantro, parsley, coriander, basildry1, basil, lemonjuice, salt, pepper], 'salsa_mixture')
         ^^^
NameError: name 'mix' is not defined. Did you mean: 'max'?
Alice mixes the large egg whites and confectioners' sugar to make some royalicing_base1.
Afterwards, Alice mixes the large egg whites and confectioners' sugar to make some leaves_icing_base1.
Afterwards, she mixes the royalicing_base1 and food coloring, brown, red and yellow to make some royalicing1.



In [11]:
# time.sleep(1)
!python 'recipe_task/recipe_task.py'
# time.sleep(1)
output_text = open("../recipe_task/output.txt", "r").read()
print(output_text)

Alice mixes the olive oil, melted butter and large egg white to make some oil_eggwhite_butter_mix.
Alice fries the olive oil, portabella mushrooms, diced onions, garlic cloves, fresh spinach, white wine, dried thyme, rosemary, salt and cracked pepper to make some stuffing.
Then, Alice mixes the stuffing and egg, lightly beaten to make some stuffing_with_egg.



In [13]:
# aug_ings = ['10 oz garam masala', '5 cloves star anise']
# aug_ings = []

test = WrongIngTestOld(output_text, transcipt_steps, selected_row, actor_specs, max_recipes=max_recipes)

model_list = ["OpenAI/gpt-oss-20B",
              "meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8",
              "Qwen/Qwen3-235B-A22B-Instruct-2507-tput",
              "moonshotai/Kimi-K2-Instruct-0905",
              "deepseek-ai/DeepSeek-R1",]

ing_acc = {}
for ing in test.unused_ingredients: ing_acc[ing] = []
overall_acc = []

for model in model_list:
    print(f'Testing {model}:')
    start = time.time()
    prompt = f'Given a transcript of actions performed by {actor_specs[0]}, output your reasoning followed by the name of the recipe {actor_specs[0]} is trying to cook.'
    test.run_test(client=Together(), model=model, prompt=prompt, n=num_recipes)
    test.print_results(match_exact=False)
    accuracies = test.get_results(match_exact=False)
    print(accuracies)

    print(f'Testing finished: ({'%.2f' % (time.time()-start)} seconds)')
    print('='*50)
    print()

    for i, ing in enumerate(test.unused_ingredients): ing_acc[ing].append(accuracies[i])
    overall_acc.append(sum(accuracies)/len(accuracies))

IndexError: list index out of range

In [6]:
model = 'gpt-5'

print(f'Testing {model}:')
start = time.time()
test.run_test(prompt=f'Given a transcript of actions performed by {actor_specs[0]}, output your reasoning followed by the name of the recipe {actor_specs[0]} is trying to cook.', n=num_recipes)
test.print_results(match_exact=False)
accuracies = test.get_results(match_exact=False)
print(accuracies)

print(f'Testing finished: ({'%.2f' % (time.time()-start)} seconds)')
print('='*50)
print()

model_list.append(model)
for i, ing in enumerate(test.unused_ingredients): ing_acc[ing].append(accuracies[i])
overall_acc.append(sum(accuracies)/len(accuracies))

Testing gpt-5:
Running recipe with 'onion', spec level 0.
[**********]
Running recipe with 'Tomato', spec level 0.
[**********]
Running recipe with 'cheddar cheese', spec level 0.
[**********]
Expected: Recipe 2
Actual: ['Reasoning:\nAlice uses green peppers, ground beef, taco seasoning, kidney beans, and salsa—common to all recipes. When she mixes in the onion, she “appears shocked,” suggesting onion was not supposed to be included. Among the options, only Recipe 2 does not include onion, so that must be the recipe she intended to follow.\n\nRecipe 2', 'Reasoning: All initial steps (parboiling peppers, browning beef, mixing with taco seasoning, kidney beans, and salsa) are common to all four recipes. The first differentiator is adding onion. Alice “appears shocked” right after mixing in the onion, suggesting onion is not supposed to be in the intended recipe. Among the options, only Recipe 2 does not include onion.\n\nRecipe 2', 'Alice became shocked right after adding onion, implying

In [7]:
d = {'model': model_list}
for ing in test.unused_ingredients: d[ing] = ing_acc[ing]
d[f'overall accuracy (n={num_recipes * (max_recipes-1)})'] = overall_acc
df = pd.DataFrame(data=d)
df

,model,onion,Tomato,cheddar cheese,overall accuracy (n=30)
0,OpenAI/gpt-oss-20B,0.2,0.0,0.0,0.066667
1,meta-llama/Llama-4-Maverick-17B-128E-Instruct-FP8,0.0,0.0,0.0,0.000000
2,Qwen/Qwen3-235B-A22B-Instruct-2507-tput,1.0,0.1,0.4,0.500000
3,moonshotai/Kimi-K2-Instruct-0905,0.4,0.7,0.3,0.466667
4,deepseek-ai/DeepSeek-R1,0.9,0.4,0.5,0.600000
5,gpt-5,1.0,0.9,0.5,0.800000


In [8]:
print('Let\'s analyze the transcript step by step and compare it to the ingredients in each recipe to determine which recipe Alice is attempting to make.\n\nTranscript actions:\n1. Alice boils the green peppers → "parboiled peppers"\n2. Alice fries the ground beef → "browned beef"\n3. Alice stirs together: browned beef, taco seasoning, kidney beans, and salsa → "meat mixture"\n4. Alice mixes in the tomato\n5. Alice appears shocked — possibly indicating something is wrong or missing\n\nNow, let’s compare the ingredients of each recipe:\n\nAll recipes include:\n- 1 lb. Ground Beef\n- 1 pkg. taco seasoning\n- 1 san Kidney beans (likely a typo for "can")\n- 1 C salsa\n- 4 green peppers\n- 1/2 C sour cream\n- 10 oz garam masala (unusual for a taco-style recipe — garam masala is Indian spice)\n- 5 cloves star anise (also unusual — these are strong, sweet spices)\n\nNow, differences:\n\nRecipe 1:\n- Onion, chopped ✅\n- Tomato, chopped ✅\n- Cheddar cheese ✅\n\nRecipe 2:\n- No onion ❌\n- Tomato, chopped ✅\n- Cheddar cheese ✅\n\nRecipe 3:\n- Onion, chopped ✅\n- No tomato ❌\n- Cheddar cheese ✅\n\nRecipe 4:\n- Onion, chopped ✅\n- Tomato, chopped ✅\n- No cheddar cheese ❌\n\nNow, in the transcript:\n- Alice adds tomato → So recipes without tomato (Recipe 3) can be ruled out.\n- She did not mention adding cheese, but the shock may indicate a missing ingredient.\n\nBut the key is: she adds the tomato, and then "appears shocked."\n\nWhy would she be shocked?\n\nLet’s look at the bizarre ingredients: garam masala and star anise in what seems like a taco recipe. That’s highly unusual. But all four recipes include these.\n\nWait — the real clue may be in the missing ingredient at the time of shock.\n\nShe adds tomato after the meat mixture. Then she’s shocked.\n\nPerhaps she realizes she’s missing a key ingredient that should have been added earlier or is essential.\n\nBut more importantly, let’s see which recipes include both onion and tomato.\n\nRecipes with both onion and tomato:\n- Recipe 1: ✅ onion, ✅ tomato, ✅ cheese\n- Recipe 4: ✅ onion, ✅ tomato, ❌ cheese\n\nRecipe 2: no onion, has tomato\nRecipe 3: has onion, no tomato\n\nIn the transcript, Alice boils green peppers, fries beef, combines with taco seasoning, beans, salsa — then adds tomato.\n\nBut what about the onion?\n\nThe transcript does **not** mention chopping or adding onion.\n\nIf the recipe requires onion, and she skipped it, that might explain her shock — realizing she forgot an ingredient.\n\nBut she only appears shocked **after adding the tomato**.\n\nSo if she has onion in her recipe, and didn’t add it, she might realize it when she gets to the tomato.\n\nBut if her recipe **doesn’t require onion**, then forgetting it wouldn’t be a problem.\n\nWait — in Recipe 2, onion is **not** in the ingredients.\n\nSo if Alice is making Recipe 2, she shouldn’t need onion.\n\nBut if she **has** onion in her recipe (Recipes 1, 3, 4), then she should have used it.\n\nBut she didn’t mention onion in the actions.\n\nSo if she is making Recipe 1, 3, or 4, she should have chopped and added onion earlier.\n\nBut she didn’t.\n\nSo why is she shocked only after adding tomato?\n\nPerhaps because she realizes she forgot the **cheese**?\n\nBut cheese is usually added at the end, as a topping.\n\nIn the transcript, she hasn’t finished — no mention of serving or final toppings.\n\nAlternatively, maybe the shock is due to tasting or realizing the recipe has bizarre spices.\n\nBut all recipes have garam masala and star anise.\n\nAnother idea: “san” is likely a typo for “can”.\n\nBut more importantly, let’s look at **which recipe is missing the tomato**.\n\nRecipe 3 does **not** have tomato.\n\nSo if Alice is making Recipe 3, and she adds tomato, that would be a mistake — adding an ingredient not in the recipe.\n\nBut she’s following the recipe — so why would she add tomato if it’s not in the recipe?\n\nUnless she **thinks** it’s in the recipe.\n\nBut the shock could come from realizing that tomato isn’t in the recipe — she checks and sees it’s not listed.\n\nBut the transcript says: "Alice mixes in the Tomato. Alice appears shocked."\n\nSo she adds it, then is shocked — possibly realizing it shouldn’t be there.\n\nThat would make sense if the recipe **doesn’t include tomato**.\n\nWhich recipe doesn’t have tomato?\n\nRecipe 3: No tomato.\n\nSo if she is making Recipe 3, and adds tomato, then sees it’s not in the ingredients, she’d be shocked.\n\nBut let’s check Recipe 3’s ingredients:\n- Ground beef\n- taco seasoning\n- kidney beans\n- onion, chopped ✅\n- salsa\n- green peppers\n- cheddar cheese ✅\n- sour cream\n- garam masala\n- star anise\n\nNo tomato.\n\nSo she adds tomato — not in recipe — then is shocked.\n\nThat fits.\n\nNow, what about onion? Recipe 3 has onion, but she didn’t mention adding it.\n\nBut the transcript doesn’t say she didn’t add it — it just doesn’t mention it.\n\nThe transcript only describes:\n- Boil peppers\n- Fry beef\n- Mix beef, seasoning, beans, salsa\n- Add tomato\n\nIt doesn’t mention onion at all.\n\nSo maybe she forgot it, but didn’t notice until later.\n\nBut the shock comes **after adding tomato**, not onion.\n\nSo the shock is likely due to the tomato, not the onion.\n\nTherefore, the most plausible explanation is:\n- She is following a recipe that **does not include tomato**\n- She adds tomato anyway (maybe out of habit)\n- Then realizes it’s not in the recipe → shock\n\nOnly Recipe 3 lacks tomato.\n\nRecipe 2 has tomato, Recipe 1 has tomato, Recipe 4 has tomato.\n\nOnly Recipe 3 does **not** have tomato.\n\nTherefore, Alice is making Recipe 3.\n\nBut wait — Recipe 2 also lacks onion.\n\nBut she didn’t add onion — so if she were making Recipe 2, no problem.\n\nBut Recipe 2 **has** tomato, so adding tomato would be correct — no reason to be shocked.\n\nSimilarly, Recipe 1 and 4 have tomato — so adding it is fine.\n\nOnly Recipe 3 does **not** have tomato — so adding it would be a mistake.\n\nHence, the shock.\n\nTherefore, the recipe Alice is trying to cook is:\n\nRecipe 3\n\nBut let’s double-check:\n\nRecipe 3 ingredients:\n- Ground beef ✅\n- taco seasoning ✅\n- kidney beans ✅\n- onion, chopped (not mentioned — possibly forgotten, but not the cause of shock)\n- salsa ✅\n- green peppers ✅\n- cheddar cheese ✅ (not added yet, but usually a topping)\n- sour cream ✅\n- garam masala ✅\n- star anise ✅\n\nNo tomato — so adding tomato is an error.\n\nShe adds it — then shock.\n\nYes.\n\nAlternatively, could the shock be because she sees the garam masala and star anise? But all recipes have them.\n\nNo distinction.\n\nSo the only recipe where adding tomato would be a mistake is Recipe 3.\n\nTherefore, despite not mentioning onion, she is likely making Recipe 3.\n\nFinal answer:\n\nRecipe 3',

'Let\'s analyze the transcript step by step and compare it to the ingredients in each recipe to determine which recipe Alice is attempting to make.\n\nTranscript Analysis:\n\n1. "Alice boils the green peppers to make some parboiled peppers."\n- This indicates she is using green peppers, which are present in all four recipes. Boiling or parboiling suggests a preparation step, possibly for stuffing or softening — common in dishes like stuffed peppers.\n\n2. "Continuing with the recipe, Alice fries the Ground Beef to make some browned beef."\n- Ground beef is in all four recipes. This step is standard in many ground beef-based recipes.\n\n3. "She stirs the browned beef, taco seasoning, Kidney beans and salsa to make some meat mixture."\n- All four recipes include: Ground Beef, taco seasoning, Kidney beans, and salsa. So far, consistent with all.\n\n4. "Alice mixes in the Tomato."\n- She adds tomato. This is important.\n  - Recipe 1: includes "1 medium Tomato, chopped" → ✅ has tomato\n  - Recipe 2: includes "1 medium Tomato, chopped" → ✅ has tomato\n  - Recipe 3: does NOT include tomato → ❌ no tomato\n  - Recipe 4: does NOT include tomato → ❌ no tomato\n\nSo after this step, only Recipes 1 and 2 are possible.\n\nNow, the key clue: "Alice appears shocked."\n\nWhy would she be shocked?\n\nLet’s look at the ingredients in Recipes 1 and 2.\n\nRecipe 1:\n- Includes: onion, tomato, green peppers, cheese, sour cream, garam masala (10 oz), star anise (5 cloves)\n\nRecipe 2:\n- Same as Recipe 1 but MISSING onion\n- Also has garam masala and star anise\n\nWait — but both recipes have garam masala and star anise, which are Indian spices.\n\nTaco seasoning + ground beef + salsa + green peppers + tomato → sounds like a Tex-Mex or taco-style dish.\n\nBut garam masala and star anise are strongly associated with Indian or Asian cuisine, not Mexican.\n\n10 oz of garam masala and 5 cloves of star anise is an extremely large amount — especially for a dish with only 1 lb of beef.\n\nNormal garam masala usage is 1–2 teaspoons, not 10 oz (which is about 280 grams — a massive quantity).\n\nSimilarly, 5 whole star anise cloves is a very strong, licorice-like flavor — not typical in a taco dish.\n\nSo Alice is likely making what she thinks is a taco or stuffed pepper recipe, but the inclusion of garam masala and star anise (in huge quantities) suggests a serious error.\n\nBut the shock comes after she adds the tomato.\n\nWhy would adding tomato cause shock?\n\nPossibility: She realizes something is wrong with the recipe — perhaps she sees the unusual spices now, or the dish looks off.\n\nBut more likely: The recipe itself is absurd. The presence of garam masala and star anise in a taco-like recipe is a red flag.\n\nBut all four recipes have garam masala and star anise.\n\nSo why is she shocked after adding tomato?\n\nLet’s compare Recipes 1 and 2 again.\n\nRecipe 1: has onion\nRecipe 2: does NOT have onion\n\nIn the transcript: there is no mention of onion being used.\n\nShe boiled peppers, browned beef, mixed beef with taco seasoning, beans, salsa, then added tomato.\n\nNo mention of chopping or adding onion.\n\nBut if the recipe calls for onion, and she forgot it, she might realize it now — but the shock would be due to forgetting an ingredient.\n\nBut the transcript says "Alice appears shocked" — not that she forgot something, but something about the action causes shock.\n\nAlternatively, perhaps the recipe itself is so bizarre (Mexican dish with Indian spices) that when she adds the tomato — a final fresh ingredient — she steps back and sees the full ingredient list, including "10 oz garam masala" and "5 cloves star anise", and realizes this can\'t be right.\n\nBut again, all recipes have those.\n\nWait — is there a typo?\n\n"1 san Kidney beans" — likely "1 can Kidney beans"\n\nBut more importantly: "10 oz garam masala" — that must be a mistake. Garam masala is a spice blend, typically used in small amounts. 10 oz is over 10 tablespoons — it should be 1 tsp or 1 oz at most.\n\nSimilarly, 5 cloves of star anise is extremely strong.\n\nSo perhaps the shock is due to realizing the recipe has absurd quantities.\n\nBut why after adding tomato?\n\nPerhaps tomato is the last ingredient before the weird spices are added.\n\nBut in the transcript, she hasn\'t added cheese, sour cream, or the spices yet.\n\nThe meat mixture was made with: beef, taco seasoning, beans, salsa.\n\nThen she added tomato.\n\nThen shock.\n\nSo she hasn’t added garam masala or star anise yet.\n\nSo she hasn’t seen them yet? But she must have read the recipe.\n\nUnless she is following the recipe step by step and hasn’t gotten to those ingredients.\n\nBut the transcript doesn’t mention adding garam masala or star anise.\n\nSo perhaps the shock is because she realizes something is missing or wrong.\n\nBut let’s look at the ingredients again.\n\nAll recipes have:\n- Ground beef\n- Taco seasoning\n- Kidney beans\n- Salsa\n- Green peppers\n- Sour cream\n- Garam masala (10 oz)\n- Star anise (5 cloves)\n\nThe differences:\n- Recipe 1: has onion and tomato and cheese\n- Recipe 2: has tomato and cheese, no onion\n- Recipe 3: has onion and cheese, no tomato\n- Recipe 4: has onion, no tomato, no cheese\n\nBut in the transcript:\n- She boiled green peppers\n- Fried beef\n- Made meat mixture with beef, taco seasoning, beans, salsa\n- Added tomato\n\nSo she used tomato.\n\nTherefore, only Recipe 1 and 2 are possible.\n\nBut Recipe 2 does not have onion.\n\nIn the transcript, there is no mention of onion.\n\nIf she were following Recipe 1, she should have chopped and added onion earlier.\n\nBut she didn’t mention it.\n\nSo if she skipped onion, and then adds tomato and realizes she forgot the onion — that might cause shock.\n\nBut Recipe 2 doesn’t have onion, so no need to add it.\n\nSo if she is following Recipe 2, she wouldn’t expect onion.\n\nBut Recipe 2 is missing onion — is that a typo?\n\nPerhaps.\n\nBut more importantly: the presence of garam masala and star anise in a taco-style recipe is bizarre.\n\nBut the key might be that "garam masala" and "star anise" are not typical taco ingredients.\n\nIn fact, they are completely out of place.\n\nSo perhaps the shock is not about missing or extra ingredients, but about realizing the recipe is a fusion or a mistake.\n\nBut we need to pick one recipe.\n\nLet’s list the ingredients actually used in the transcript:\n\n- Green peppers: used (boiled)\n- Ground beef: used\n- Taco seasoning: used\n- Kidney beans: used\n- Salsa: used\n- Tomato: used\n\nCheese and sour cream are not mentioned as used yet.\n\nBut sour cream is in all recipes.\n\nCheese: in Recipes 1, 2, 3 — not in Recipe 4.\n\nBut she hasn’t added cheese.\n\nThe shock happens after adding tomato.\n\nNow, here’s a possibility: when she adds the tomato, she sees the recipe and notices "10 oz garam masala" and "5 cloves star anise" — ingredients that are completely incompatible with a tomato-based taco dish.\n\nShe realizes this can’t be right — hence the shock.\n\nBut again, all recipes have those.\n\nSo why is this a clue?\n\nWait — perhaps the recipe she is using is not a real recipe, but a mix-up.\n\nBut we have to choose among the four.\n\nAnother idea: "1 san Kidney beans" — likely "1 can", but "san" might be typo for "can".\n\nBut more importantly, let’s look at what’s unique.\n\nRecipe 1 has: onion, tomato, cheese\n\nRecipe 2 has: tomato, cheese, no onion\n\nBut in the transcript, no onion is used.\n\nIf she were making Recipe 1, she should have used onion.\n\nBut she didn’t.\n\nSo she is likely not making Recipe 1.\n\nTherefore, she is making Recipe 2, which doesn’t have onion.\n\nBut Recipe 2 has tomato — which she added.\n\nAnd she didn’t need onion.\n\nSo that fits.\n\nBut why is she shocked?\n\nPerhaps because she now sees the next ingredients: garam masala and star anise — which are absurd for this dish.\n\nBut again, all recipes have them.\n\nUnless the shock is not about the spices, but about something else.\n\nWait — Recipe 3 and 4 don’t have tomato, but she added tomato.\n\nSo only Recipe 1 and 2 call for tomato.\n\nRecipe 1 requires onion — she didn’t use it.\n\nRecipe 2 does not require onion — she didn’t use it.\n\nSo Recipe 2 is the only one consistent with the actions: she used tomato, didn’t use onion.\n\nTherefore, she is following Recipe 2.\n\nBut why the shock?\n\nBecause after adding tomato, she looks at the remaining ingredients: "10 oz garam masala", "5 cloves star anise" — which are clearly wrong for a taco-stuffed pepper dish.\n\nShe realizes the recipe is corrupted or a prank.\n\nBut the question is: which recipe is she trying to cook?\n\nBased on the ingredients she used and didn’t use, Recipe ')

Let's analyze the transcript step by step and compare it to the ingredients in each recipe to determine which recipe Alice is attempting to make.

Transcript actions:
1. Alice boils the green peppers → "parboiled peppers"
2. Alice fries the ground beef → "browned beef"
3. Alice stirs together: browned beef, taco seasoning, kidney beans, and salsa → "meat mixture"
4. Alice mixes in the tomato
5. Alice appears shocked — possibly indicating something is wrong or missing

Now, let’s compare the ingredients of each recipe:

All recipes include:
- 1 lb. Ground Beef
- 1 pkg. taco seasoning
- 1 san Kidney beans (likely a typo for "can")
- 1 C salsa
- 4 green peppers
- 1/2 C sour cream
- 10 oz garam masala (unusual for a taco-style recipe — garam masala is Indian spice)
- 5 cloves star anise (also unusual — these are strong, sweet spices)

Now, differences:

Recipe 1:
- Onion, chopped ✅
- Tomato, chopped ✅
- Cheddar cheese ✅

Recipe 2:
- No onion ❌
- Tomato, chopped ✅
- Cheddar cheese ✅

Recipe

In [1]:
print('**Reasoning**\n\n1. **Ingredients Alice actively uses in the transcript**  \n   - *Ground Beef* – she fries it.  \n   - *Taco seasoning* – she mixes it.  \n   - *Kidney beans* – she mixes it.  \n   - *Salsa* – she mixes it.  \n   - *Green peppers* – she boils them.  \n   - *Cheddar cheese* – she adds it.  \n\n   These ten ingredients are all present in **recipe\u202f2**.\n\n2. **Checking the four recipe lists**\n\n| Recipe | 1 lb. Ground Beef | Taco seasoning | Kidney beans | Green peppers | Onion | Tomato | Cheddar | Sour cream | Garam\u202fmasala | Star anise |\n|--------|-------------------|----------------|--------------|---------------|-------|--------|---------|------------|--------------|------------|\n| **1**  | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ |\n| **2**  | ✔ | ✔ | ✔ | ✔ | ✗ | ✔ | ✔ | ✔ | ✔ | ✔ |\n| **3**  | ✔ | ✔ | ✔ | ✔ | ✔ | ✗ | ✔ | ✔ | ✔ | ✔ |\n| **4**  | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ | ✗ | ✗ | ✗ | ✗ |\n\n*Recipe\u202f2* is the only one that:\n- **Includes all ingredients Alice has already used** (ground beef, seasoning, beans, salsa, peppers, cheddar).  \n- **Does not require onion** – she never mentioned using it (recipe\u202f2 omits it).  \n- The remaining ingredients (sour cream, garam\u202fmasala, star anise, tomato) are present but she hasn’t mentioned them yet; that’s consistent with the transcript stopping early.\n\n3. **Conclusion**  \n   The transcript of Alice’s actions matches **Recipe\u202f2**.\n\n**Recipe Alice is trying to cook: Recipe\u202f2**')

**Reasoning**

1. **Ingredients Alice actively uses in the transcript**  
   - *Ground Beef* – she fries it.  
   - *Taco seasoning* – she mixes it.  
   - *Kidney beans* – she mixes it.  
   - *Salsa* – she mixes it.  
   - *Green peppers* – she boils them.  
   - *Cheddar cheese* – she adds it.  

   These ten ingredients are all present in **recipe 2**.

2. **Checking the four recipe lists**

| Recipe | 1 lb. Ground Beef | Taco seasoning | Kidney beans | Green peppers | Onion | Tomato | Cheddar | Sour cream | Garam masala | Star anise |
|--------|-------------------|----------------|--------------|---------------|-------|--------|---------|------------|--------------|------------|
| **1**  | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ |
| **2**  | ✔ | ✔ | ✔ | ✔ | ✗ | ✔ | ✔ | ✔ | ✔ | ✔ |
| **3**  | ✔ | ✔ | ✔ | ✔ | ✔ | ✗ | ✔ | ✔ | ✔ | ✔ |
| **4**  | ✔ | ✔ | ✔ | ✔ | ✔ | ✔ | ✗ | ✗ | ✗ | ✗ |

*Recipe 2* is the only one that:
- **Includes all ingredients Alice has already used** (ground beef, s

- Transcript length
- \# Options: 2,4,8,16,32
- \# Ingredients: 2,4,8,16,32